# IVID — Train YOLOv8 trên Google Colab

Notebook này train model phát hiện lỗi bề mặt thép (NEU-DET) cho dự án
**Industrial Visual Inspection & Deployment Benchmark**.

**Cách dùng:** chạy từng ô theo thứ tự, bấm ▶ ở góc trái mỗi ô (hoặc `Shift+Enter`).
Đừng bỏ qua ô nào. Nếu một ô báo lỗi, dừng lại và hỏi trước khi chạy tiếp.

---

### ⚠️ Ba điều phải biết về Colab

| Sự thật | Hậu quả | Notebook này xử lý thế nào |
|---|---|---|
| Máy ảo bị **xoá sạch** khi ngắt kết nối | Mất hết kết quả | Ghi thẳng vào Google Drive |
| Ngắt sau ~90 phút không thao tác | Train dở dang | Có ô `resume` ở Phần 8 |
| GPU miễn phí có hạn mức | Không xin được GPU | Train YOLOv8n trước, YOLOv8s sau |

**Đừng đóng tab. Thỉnh thoảng bấm vào trang để Colab biết bạn còn ở đó.**

---
## Phần 1 — Xin GPU

**Làm bằng tay trước khi chạy ô dưới:**

Menu **Runtime → Change runtime type** → mục **Hardware accelerator** chọn **T4 GPU** → **Save**.

Đây là bước dễ quên nhất. Quên thì train bằng CPU, chậm gấp ~50 lần.

In [ ]:
import subprocess, sys

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or
      'KHONG THAY GPU')

import torch
print('torch      :', torch.__version__)
print('CUDA co    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))
    print('VRAM       : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print()
    print('=' * 60)
    print('DUNG LAI. Chua bat GPU.')
    print('Runtime -> Change runtime type -> T4 GPU -> Save, roi chay lai o nay.')
    print('=' * 60)

---
## Phần 2 — Nối Google Drive

Sẽ hiện popup xin quyền → bấm **Cho phép** → chọn tài khoản Google của bạn.

Mọi thứ ghi vào `/content/drive/MyDrive/ivid` sẽ **sống sót qua mọi lần ngắt kết nối**.
Mọi thứ ghi ở chỗ khác sẽ mất.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE = '/content/drive/MyDrive/ivid'
os.makedirs(f'{SAVE}/runs', exist_ok=True)
os.makedirs(f'{SAVE}/artifacts', exist_ok=True)
print('Se luu vao:', SAVE)

---
## Phần 3 — Lấy code

Repo đang để **private**, nên `git clone` bình thường sẽ báo lỗi. Có hai cách:

**Cách A — đổi repo sang public** (đơn giản nhất, và dự án này rồi cũng sẽ public
vì bạn dùng nó để ứng tuyển):
vào `github.com/quocanh112233/industrial-visual-inspection` → **Settings** →
kéo xuống cuối → **Change visibility** → **Make public**.

**Cách B — dùng token** (giữ repo private):
vào https://github.com/settings/tokens?type=beta → **Generate new token** →
chọn repo này → quyền **Contents: Read-only** → tạo và copy token.
Ô dưới sẽ hỏi token; **nó không bị lưu vào notebook**, gõ xong là xong.

Chạy ô dưới — nó tự thử cách A trước, không được thì hỏi token.

In [ ]:
%cd /content
import subprocess, getpass, shutil, os

REPO = 'quocanh112233/industrial-visual-inspection'
DIR  = '/content/industrial-visual-inspection'
shutil.rmtree(DIR, ignore_errors=True)

def clone(url, show):
    r = subprocess.run(['git', 'clone', '--depth', '1', url, DIR],
                       capture_output=True, text=True)
    print(('OK  ' if r.returncode == 0 else 'THAT BAI  ') + show)
    if r.returncode != 0:
        print('   ', r.stderr.strip().splitlines()[-1] if r.stderr.strip() else '')
    return r.returncode == 0

ok = clone(f'https://github.com/{REPO}.git', 'clone cong khai')
if not ok:
    print()
    print('Repo dang private. Dan token vao o ben duoi (khong hien ra man hinh):')
    tok = getpass.getpass('GitHub token: ').strip()
    ok = clone(f'https://{tok}@github.com/{REPO}.git', 'clone bang token')
    del tok

assert ok, 'Khong clone duoc. Xem lai Cach A hoac Cach B o tren.'
%cd $DIR
!git log --oneline -1
!ls

In [ ]:
# Colab da co san torch voi CUDA, chi can them ultralytics.
# KHONG can ghim numpy nhu tren Jetson — rang buoc do chi ap dung cho JetPack.
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

---
## Phần 4 — Dữ liệu

Dùng đúng script mà bạn đã chạy trên Jetson, nên cách chia dữ liệu giống hệt.

In [ ]:
!bash scripts/download_dataset.sh
!PYTHONPATH=src python -m ivid.data.prepare  --config configs/data.yaml
!PYTHONPATH=src python -m ivid.data.validate --config configs/data.yaml
!PYTHONPATH=src python -m ivid.data.stats    --config configs/data.yaml

In [ ]:
# Doi chieu voi Jetson: hash phai GIONG HET, neu khong thi hai may dang
# train/do tren hai tap du lieu khac nhau va moi so sanh deu vo nghia.
import json
JETSON_SHA = '757d9bccc4fbc4fada9b28078b938d0f4dca9327f3840c7cfc214be236efefa0'
m = json.load(open('results/dataset_manifest.json'))
print('hash Colab :', m['dataset_sha256'])
print('hash Jetson:', JETSON_SHA)
print()
if m['dataset_sha256'] == JETSON_SHA:
    print('KHOP — cung mot tap du lieu, cung mot cach chia.')
else:
    print('KHONG KHOP — DUNG LAI va bao lai truoc khi train.')
print()
print('so anh :', m['counts'])
print('so bbox:', m['boxes'])

---
## Phần 5 — Train YOLOv8n

Khoảng **40–50 phút** trên T4. Kết quả ghi thẳng vào Drive nên mất kết nối cũng không mất gì.

Trong lúc chạy bạn sẽ thấy bảng tiến độ từng epoch với `box_loss`, `cls_loss`, `mAP50`.
Con số `mAP50` ở cột phải nên tăng dần và ổn định quanh 0.7–0.8.

In [ ]:
!PYTHONPATH=src python -m ivid.train.train \
    --config configs/train_yolov8n.yaml \
    --project /content/drive/MyDrive/ivid/runs

### Đánh giá + sao lưu — **đừng bỏ qua ô này**

`best.pt` đã nằm trong Drive nhờ `--project`, nhưng `results/` thì vẫn ở trong máy ảo.
Đó là nơi chứa `train_yolov8n_manifest.json` — file ghi seed, phiên bản 8 thư viện,
siêu tham số và hash dataset. Mất nó là mất toàn bộ bằng chứng tái lập (FR-06),
và con số mAP trong báo cáo thành con số không truy nguyên được.

In [ ]:
!PYTHONPATH=src python -m ivid.train.evaluate --name yolov8n

!mkdir -p /content/drive/MyDrive/ivid/artifacts
!cp -r results models /content/drive/MyDrive/ivid/artifacts/
!ls -lh /content/drive/MyDrive/ivid/artifacts/models/yolov8n/
!ls -l  /content/drive/MyDrive/ivid/artifacts/results/

In [ ]:
# Xem ky ket qua theo tung lop
import json
d = json.load(open('results/train_eval.json'))
for name, v in d.items():
    o = v['overall']
    print(f"{name}")
    print(f"  mAP@0.5      {o['mAP50']:.4f}   (nguong FR-05: >= 0.65)")
    print(f"  mAP@0.5:0.95 {o['mAP50_95']:.4f}")
    print(f"  precision    {o['precision']:.4f}")
    print(f"  recall       {o['recall']:.4f}")
    print()
    print(f"  {'lop':<18}{'mAP50':>9}{'mAP50-95':>11}")
    for c, mm in v['per_class'].items():
        print(f"  {c:<18}{mm.get('mAP50', 0):>9.4f}{mm.get('mAP50_95', 0):>11.4f}")

print()
print('Neu mAP@0.5 < 0.65: xem cot theo lop TRUOC khi ket luan la hong.')
print('pitted_surface (khung phu > 50% dien tich anh) va crazing (vet nut lan toa,')
print('khong co bien ro) von kho — mot minh chung keo xuong la binh thuong.')

---
## Phần 6 — Train YOLOv8s

Khoảng **1.5–2 tiếng**. Model lớn hơn, mAP cao hơn, chậm hơn — đây là chiều so sánh
thứ hai của bảng benchmark.

> Nếu Colab đã chạy được hơn 2 tiếng, **nên dừng ở đây**, tải YOLOv8n về, rồi quay lại
> làm YOLOv8s trong một phiên mới. Phiên miễn phí thường bị cắt quanh mốc 4 tiếng.

In [ ]:
!PYTHONPATH=src python -m ivid.train.train \
    --config configs/train_yolov8s.yaml \
    --project /content/drive/MyDrive/ivid/runs

In [ ]:
!PYTHONPATH=src python -m ivid.train.evaluate --name yolov8s
!cp -r results models /content/drive/MyDrive/ivid/artifacts/
!ls -lh /content/drive/MyDrive/ivid/artifacts/models/*/

---
## Phần 7 — Tải về máy

Ô dưới gói mọi thứ cần mang sang Jetson thành một file zip nhỏ (~30 MB).

In [ ]:
import shutil, os
os.makedirs('/content/ivid_out', exist_ok=True)
!cp -r /content/drive/MyDrive/ivid/artifacts/models  /content/ivid_out/
!cp -r /content/drive/MyDrive/ivid/artifacts/results /content/ivid_out/
shutil.make_archive('/content/ivid_models', 'zip', '/content/ivid_out')
print('%.1f MB' % (os.path.getsize('/content/ivid_models.zip')/1e6))

from google.colab import files
files.download('/content/ivid_models.zip')

### Đưa lên Jetson

Chạy trên **máy dev** (không phải trong Colab), sau khi file đã về `~/Downloads`:

```bash
cd ~/Downloads && unzip -o ivid_models.zip -d ivid_models
JET=jetson@<IP-JETSON>
DST=~/quoc_anh/industrial-visual-inspection

scp -r ivid_models/models/*  $JET:$DST/models/
scp -r ivid_models/results/* $JET:$DST/results/
```

Rồi trên **Jetson**:

```bash
cd ~/quoc_anh/industrial-visual-inspection
make export-onnx MODEL=yolov8n && make export-trt MODEL=yolov8n
make export-onnx MODEL=yolov8s && make export-trt MODEL=yolov8s
make parity MODEL=yolov8n

sudo nvpmodel -m 0 && sudo jetson_clocks   # co dinh dieu kien do (C3)
make bench && make accuracy && make report
```

---
## Phần 8 — Nếu bị ngắt giữa chừng

Chuyện bình thường, không phải hỏng. Bấm **Connect** để kết nối lại, chạy lại
**Phần 2** (mount Drive), **Phần 3** (clone + cài), rồi chạy ô dưới.

Nó chạy tiếp từ epoch dang dở, không train lại từ đầu.

In [ ]:
MODEL = 'yolov8n'      # doi thanh 'yolov8s' neu dang train ban s

from ultralytics import YOLO
last = f'/content/drive/MyDrive/ivid/runs/{MODEL}/weights/last.pt'
import os
assert os.path.exists(last), f'khong thay {last} — chua train lan nao?'
YOLO(last).train(resume=True)

---
## Lỗi thường gặp

| Triệu chứng | Nguyên nhân | Xử lý |
|---|---|---|
| `nvidia-smi: command not found` | Chưa bật GPU | Runtime → Change runtime type → T4 GPU |
| `GPU unavailable` | Hết hạn mức GPU miễn phí | Chờ vài giờ, hoặc dùng Kaggle Notebooks (30h GPU/tuần) |
| `Repository not found` khi clone | Repo private, token sai/thiếu quyền | Xem Phần 3, cách A hoặc B |
| Mất hết file sau khi quay lại | Không lưu vào Drive | Luôn dùng `--project /content/drive/...` |
| `CUDA out of memory` | Batch quá lớn | Giảm `batch` trong `configs/train_yolov8*.yaml` |
| Hash dataset không khớp Jetson | Hai máy tải nguồn khác nhau | Dừng lại, báo trước khi train |